# Notebook 04 — Práctica

Cuarto y último sub-bloque del Tema 05. **14 ejercicios graduales** que combinan las tres familias de funciones que viste en los notebooks anteriores: agregadas, de strings y de fechas.

Cada ejercicio sigue el mismo patrón:

1. **Enunciado** en markdown.
2. **Celda vacía** para que escribas tu solución.
3. **Solución** colapsable (`<details>`) — ábrela solo cuando hayas intentado.

Tres niveles:

- **Fácil (1-5):** una sola familia de función a la vez.
- **Medio (6-10):** combinan agregadas con strings o fechas.
- **Difícil (11-14):** tres familias mezcladas, `HAVING`, `FILTER`.

**Contenido de este notebook:**

- [Setup](#setup)
- [Nivel fácil (1-5)](#nivel-fácil-1-5)
- [Nivel medio (6-10)](#nivel-medio-6-10)
- [Nivel difícil (11-14)](#nivel-difícil-11-14)

## Setup

In [ ]:
# Setup — instala JupySQL si hace falta (Colab trae ipython-sql, no JupySQL).
import importlib.util, subprocess, sys
if importlib.util.find_spec("jupysql") is None:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "ipython-sql"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupysql"], check=True)
    print("⚠ JupySQL instalado. REINICIA el kernel (Entorno de ejecución → Reiniciar sesión)")
    print("  y vuelve a correr esta celda y las siguientes.")
else:
    print("✓ JupySQL listo.")

In [ ]:
%load_ext sql

from sqlalchemy import create_engine

# Reemplaza con tus valores del Tema 01
AURORA_HOST     = "aurora-mod4.cluster-xxxxx.us-east-1.rds.amazonaws.com"
AURORA_PASSWORD = "TU_PASSWORD_AQUI"
AURORA_DATABASE = "northwind"

engine = create_engine(
    f"postgresql+psycopg2://postgres:{AURORA_PASSWORD}@{AURORA_HOST}:5432/{AURORA_DATABASE}"
)

%sql engine
# Devuelve cada query como DataFrame de pandas (mejor render en Colab, y
# el resultado es directamente manipulable con pandas).
%config SqlMagic.autopandas = True

## Nivel fácil (1-5)

### Ejercicio 1 — Productos por categoría

Cuenta cuántos productos hay en cada categoría de `dim_product`. Ordena de mayor a menor.

In [ ]:
%%sql
-- Tu solución aquí

<details>
<summary><strong>Solución</strong></summary>

```sql
SELECT
    category_name,
    COUNT(*) AS productos
FROM     northwind_dwh.dim_product
GROUP BY category_name
ORDER BY productos DESC;
```

`COUNT(*)` + `GROUP BY`. Es el patrón más básico de agregación.
</details>

### Ejercicio 2 — Top 5 clientes por compras

Encuentra los 5 clientes (`company_name`) con mayor monto total de compras (suma de `line_total`). Redondea a 2 decimales.

In [ ]:
%%sql
-- Tu solución aquí

<details>
<summary><strong>Solución</strong></summary>

```sql
SELECT
    dc.company_name,
    ROUND(SUM(fs.line_total), 2) AS ventas_netas
FROM      northwind_dwh.fact_sales fs
JOIN      northwind_dwh.dim_customer dc USING (customer_key)
GROUP BY  dc.company_name
ORDER BY  ventas_netas DESC
LIMIT 5;
```

El `USING (customer_key)` reemplaza al `ON fs.customer_key = dc.customer_key` cuando las columnas se llaman igual en ambos lados.
</details>

### Ejercicio 3 — Pedidos por mes en 1997

Cuenta cuántas líneas de venta hubo en cada mes de 1997. El resultado debe tener una fila por mes, ordenada cronológicamente.

In [ ]:
%%sql
-- Tu solución aquí

<details>
<summary><strong>Solución</strong></summary>

```sql
SELECT
    dd.month_number,
    dd.month_name,
    COUNT(*) AS lineas
FROM      northwind_dwh.fact_sales fs
JOIN      northwind_dwh.dim_date   dd ON dd.date_key = fs.order_date_key
WHERE     dd.year = 1997
GROUP BY  dd.month_number, dd.month_name
ORDER BY  dd.month_number;
```

Usando las columnas pre-calculadas de `dim_date`. Si no tuvieras `dim_date`, sería `EXTRACT(month FROM ...)`.
</details>

### Ejercicio 4 — Productos con 'cheese' en el nombre

Encuentra todos los productos cuyo `product_name` contenga la palabra `'cheese'` (sin importar mayúsculas/minúsculas). Devuelve `product_id`, `product_name` y `category_name`.

In [ ]:
%%sql
-- Tu solución aquí

<details>
<summary><strong>Solución</strong></summary>

```sql
SELECT
    product_id, product_name, category_name
FROM     northwind_dwh.dim_product
WHERE    product_name ILIKE '%cheese%'
ORDER BY product_name;
```

`ILIKE` para case-insensitive. Los `%` son wildcards de "cualquier cosa, incluso vacío".
</details>

### Ejercicio 5 — Listings de Airbnb por tipo

Cuenta cuántos listings hay en `airbnb.listings` por cada `room_type`. Ordena de mayor a menor.

In [ ]:
%%sql
-- Tu solución aquí

<details>
<summary><strong>Solución</strong></summary>

```sql
SELECT
    room_type,
    COUNT(*) AS listings
FROM     airbnb.listings
GROUP BY room_type
ORDER BY listings DESC;
```

Las columnas de Airbnb son `TEXT` pero `GROUP BY` y `COUNT` funcionan igual sobre texto. Para Inside Airbnb los tipos típicos son `Entire home/apt`, `Private room`, `Shared room`, `Hotel room`.
</details>

## Nivel medio (6-10)

### Ejercicio 6 — Ventas por trimestre, solo días laborables

Calcula las ventas netas por trimestre, **excluyendo sábados y domingos**. Devuelve año, trimestre y ventas netas redondeadas.

In [ ]:
%%sql
-- Tu solución aquí

<details>
<summary><strong>Solución</strong></summary>

```sql
SELECT
    dd.year,
    dd.quarter,
    ROUND(SUM(fs.line_total), 2) AS ventas_netas
FROM      northwind_dwh.fact_sales fs
JOIN      northwind_dwh.dim_date   dd ON dd.date_key = fs.order_date_key
WHERE     NOT dd.is_weekend
GROUP BY  dd.year, dd.quarter
ORDER BY  dd.year, dd.quarter;
```

`NOT dd.is_weekend` aprovecha la columna pre-calculada — más limpio que `EXTRACT(isodow FROM ...) NOT IN (6, 7)`.
</details>

### Ejercicio 7 — Empleados con ventas > $50K USD en 1997

Lista los empleados (`full_name`) cuyas ventas netas en 1997 superaron los $50,000 USD. Devuelve nombre y ventas, ordenado por ventas descendente.

In [ ]:
%%sql
-- Tu solución aquí

<details>
<summary><strong>Solución</strong></summary>

```sql
SELECT
    de.full_name,
    ROUND(SUM(fs.line_total), 2) AS ventas_netas
FROM      northwind_dwh.fact_sales fs
JOIN      northwind_dwh.dim_employee de USING (employee_key)
JOIN      northwind_dwh.dim_date     dd ON dd.date_key = fs.order_date_key
WHERE     dd.year = 1997                  -- filtro a nivel fila (WHERE)
GROUP BY  de.full_name
HAVING    SUM(fs.line_total) > 50000      -- filtro a nivel grupo (HAVING)
ORDER BY  ventas_netas DESC;
```

Combina `WHERE` (antes de agregar) y `HAVING` (después de agregar). Si tratas de poner `SUM(...) > 50000` en el `WHERE`, falla porque `SUM` aún no se ha calculado en ese paso del pipeline.
</details>

### Ejercicio 8 — Tiempo promedio de envío por shipper

Calcula el tiempo promedio (en días) entre `order_date` y `shipped_date` para cada `dim_shipper.company_name`. Ignora pedidos no enviados.

In [ ]:
%%sql
-- Tu solución aquí

<details>
<summary><strong>Solución</strong></summary>

```sql
SELECT
    ds.company_name,
    COUNT(*)                                                AS lineas_enviadas,
    ROUND(CAST(AVG(d_shp.full_date - d_ord.full_date) AS NUMERIC), 1) AS dias_promedio
FROM      northwind_dwh.fact_sales fs
JOIN      northwind_dwh.dim_shipper ds    USING (shipper_key)
JOIN      northwind_dwh.dim_date    d_ord ON d_ord.date_key = fs.order_date_key
JOIN      northwind_dwh.dim_date    d_shp ON d_shp.date_key = fs.shipped_date_key
GROUP BY  ds.company_name
ORDER BY  dias_promedio;
```

Role-playing en acción: `d_ord` y `d_shp` son dos aliases del mismo `dim_date`. El `INNER JOIN` con `d_shp` excluye automáticamente los pedidos no enviados (cuyo `shipped_date_key` es `NULL`).
</details>

### Ejercicio 9 — Precio promedio de Airbnb por alcaldía y tipo

Calcula el precio promedio de Airbnb por combinación de alcaldía (`neighbourhood_cleansed`) y `room_type`. Recuerda que `price` viene como texto con formato `'$1,234.00'`. Devuelve solo las primeras 10 filas ordenadas por precio promedio descendente.

In [ ]:
%%sql
-- Tu solución aquí

<details>
<summary><strong>Solución</strong></summary>

```sql
SELECT
    neighbourhood_cleansed                                          AS alcaldia,
    room_type,
    COUNT(*)                                                        AS listings,
    ROUND(AVG(CAST(NULLIF(REGEXP_REPLACE(price, '[$,]', '', 'g'), '') AS NUMERIC)), 2)  AS precio_promedio
FROM     airbnb.listings
WHERE    price IS NOT NULL
GROUP BY neighbourhood_cleansed, room_type
ORDER BY precio_promedio DESC
LIMIT 10;
```

`REGEXP_REPLACE(price, '[$,]', '', 'g')` quita `$` y `,` de un golpe; `NULLIF(..., '')` convierte vacíos a `NULL` para que el cast no falle en filas con `price = ''`; `CAST(... AS NUMERIC)` castea; `AVG` agrega.
</details>

### Ejercicio 10 — Listings con 'view' o 'vista' en el nombre

En `airbnb.listings`, encuentra los listings cuyo `name` contenga `'view'` o `'vista'` (sin importar capitalización). Devuelve cuántos hay y su precio promedio.

In [ ]:
%%sql
-- Tu solución aquí

<details>
<summary><strong>Solución</strong></summary>

```sql
SELECT
    COUNT(*)                                                        AS listings_con_vista,
    ROUND(AVG(CAST(NULLIF(REGEXP_REPLACE(price, '[$,]', '', 'g'), '') AS NUMERIC)), 2)  AS precio_promedio
FROM     airbnb.listings
WHERE    name ILIKE '%view%' OR name ILIKE '%vista%'
  AND    price IS NOT NULL;
```

Dos `ILIKE` unidos con `OR`. Alternativa equivalente: `name ~* 'view|vista'` (regex con flag insensible a mayúsculas).
</details>

## Nivel difícil (11-14)

### Ejercicio 11 — Ventas por día de la semana con % del total

Calcula las ventas netas por día de la semana (lunes a domingo) **y** el porcentaje que representa cada día del total general. Ordena por día de la semana (lunes primero).

In [ ]:
%%sql
-- Tu solución aquí

<details>
<summary><strong>Solución</strong></summary>

```sql
SELECT
    dd.day_of_week_number,
    dd.day_of_week_name,
    ROUND(SUM(fs.line_total), 2) AS ventas,
    ROUND(
        SUM(fs.line_total) * 100.0 /
        (SELECT SUM(line_total) FROM northwind_dwh.fact_sales),
        2
    ) AS pct_del_total
FROM      northwind_dwh.fact_sales fs
JOIN      northwind_dwh.dim_date dd ON dd.date_key = fs.order_date_key
GROUP BY  dd.day_of_week_number, dd.day_of_week_name
ORDER BY  dd.day_of_week_number;
```

El `(SELECT SUM(line_total) FROM ...)` es una **subquery escalar** — devuelve un solo valor (el gran total) y se puede usar dentro del SELECT. `* 100.0` (no `100`) fuerza división en `NUMERIC`, no entera.
</details>

### Ejercicio 12 — Managers con dos o más subordinados

Lista los managers que tienen 2 o más subordinados directos. Usa la columna `reports_to_name` de `dim_employee`. Devuelve el nombre del manager y cuántos subordinados tiene.

In [ ]:
%%sql
-- Tu solución aquí

<details>
<summary><strong>Solución</strong></summary>

```sql
SELECT
    reports_to_name AS manager,
    COUNT(*)        AS subordinados,
    STRING_AGG(full_name, ', ' ORDER BY full_name) AS quienes
FROM     northwind_dwh.dim_employee
WHERE    reports_to_name IS NOT NULL
GROUP BY reports_to_name
HAVING   COUNT(*) >= 2
ORDER BY subordinados DESC;
```

`WHERE` excluye al CEO (sin manager) antes de agregar; `HAVING` filtra a managers con ≥ 2 subordinados después; `STRING_AGG` muestra los nombres concatenados — útil para reportes.
</details>

### Ejercicio 13 — % de listings de alta respuesta por alcaldía

En Airbnb, calcula el porcentaje de listings con `host_response_rate` ≥ 90% por alcaldía (`neighbourhood_cleansed`). Excluye los `'N/A'`. Usa el modificador `FILTER` de PostgreSQL.

In [ ]:
%%sql
-- Tu solución aquí

<details>
<summary><strong>Solución</strong></summary>

```sql
SELECT
    neighbourhood_cleansed AS alcaldia,
    COUNT(*)               AS listings_con_rate,
    COUNT(*) FILTER (
        WHERE CAST(REPLACE(host_response_rate, '%', '') AS NUMERIC) >= 90
    )                      AS alta_respuesta,
    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE CAST(REPLACE(host_response_rate, '%', '') AS NUMERIC) >= 90
        ) / COUNT(*),
        1
    ) AS pct_alta
FROM     airbnb.listings
WHERE    host_response_rate IS NOT NULL
  AND    host_response_rate != 'N/A'
  AND    host_response_rate <> ''
GROUP BY neighbourhood_cleansed
ORDER BY pct_alta DESC
LIMIT 10;
```

**`COUNT(*) FILTER (WHERE …)`** es una agregación condicional limpia. Equivalente verboso: `SUM(CASE WHEN ... THEN 1 ELSE 0 END)`. `FILTER` es más legible y PostgreSQL la implementa nativamente.
</details>

### Ejercicio 14 — Estadísticas de longitud de `name` por tipo

En `airbnb.listings`, para cada `room_type` calcula: total de listings, cuántos tienen `name` con más de 50 caracteres, longitud mínima, máxima y promedio del `name`.

In [ ]:
%%sql
-- Tu solución aquí

<details>
<summary><strong>Solución</strong></summary>

```sql
SELECT
    room_type,
    COUNT(*)                                       AS total,
    COUNT(*) FILTER (WHERE LENGTH(name) > 50)      AS con_name_largo,
    MIN(LENGTH(name))                              AS min_chars,
    MAX(LENGTH(name))                              AS max_chars,
    ROUND(AVG(LENGTH(name)), 1)                    AS avg_chars
FROM     airbnb.listings
WHERE    name IS NOT NULL
GROUP BY room_type
ORDER BY total DESC;
```

Combina función de string (`LENGTH`) con varias agregadas (`COUNT`, `MIN`, `MAX`, `AVG`) y conteo condicional con `FILTER`. Los hosts de Airbnb suelen meter nombres descriptivos largos ("Cozy apt in heart of Roma Norte with rooftop view") — verás `avg_chars` típicamente entre 35 y 50.
</details>

## Cierre — y con esto cierras el Tema 05

Si llegaste aquí con las 14 soluciones ejecutadas y entendidas, **ya tienes las bases sólidas de SQL analítico**. Los Temas 06-09 amplían esas bases:

| Tema | Qué agrega sobre lo de este Tema |
|---|---|
| **06 — PL/pgSQL** | Lógica procedural en el motor: procedimientos, cursores, control de flujo |
| **07 — Funciones de ventana** | Cálculos por fila respecto a otras filas: rankings, totales móviles, comparativas |
| **08 — CTEs** | `WITH`, `WITH RECURSIVE` y análisis jerárquico para queries legibles y recursivas |
| **09 — Datos semi-estructurados** | `hstore` y `JSONB` para columnas como `amenities` y `host_verifications` de Airbnb |

**Patrón que reconocerás:** las funciones que viste son **predefinidas** — agregadas, scalar, de fechas, de strings. Las del Tema 07 (window functions) son una **categoría aparte** porque operan sobre *ventanas* de filas, no sobre grupos colapsados. Esa distinción es la que cambia lo que es posible expresar en una sola query.

---

<p align="center">
<a href="03_funciones_de_fechas.ipynb">← Anterior: Notebook 03</a> | <a href="Readme.md">Volver al índice</a> | <a href="../Tema-06/Readme.md">Siguiente: Tema 06 →</a>
</p>